In [14]:
import numpy as np
import pandas as pd
from collections import Counter
import time
from sklearn.datasets import load_iris
from sklearn.model_selection import train_test_split


In [15]:

class Node:
    def __init__(self, feature_idx=None, threshold=None, gain=None, left=None, right=None, value=None):
        # Decision Node
        self.feature_idx = feature_idx
        self.threshold = threshold
        self.gain = gain
        self.left = left
        self.right = right
        # Leaf Node
        self.value = value

In [21]:
class DecisionTree:
    def __init__(self, min_samples_split=2, max_depth=5, algorithm='CART'):
        self.min_samples_split = min_samples_split
        self.max_depth = max_depth
        self.algorithm = algorithm # 'ID3', 'C4.5', or 'CART'
        self.root = None

    def build_tree(self, dataset, curr_depth=0):
        X, y = dataset[:, :-1], dataset[:, -1]
        n_samples, n_features = X.shape

        if n_samples >= self.min_samples_split and curr_depth < self.max_depth:
            best_split = self.get_best_split(dataset, n_features)

            if best_split["gain"] > 0:
                left_node = self.build_tree(best_split["left_dataset"], curr_depth + 1)
                right_node = self.build_tree(best_split["right_dataset"], curr_depth + 1)
                return Node(best_split["feature_idx"], best_split["threshold"], 
                            best_split["gain"], left_node, right_node)

        leaf_value = Counter(y).most_common(1)[0][0]
        return Node(value=leaf_value)

    def get_best_split(self, dataset, n_features):
        best_split = {'gain': -1}
        max_gain = -float("inf")

        for feature_idx in range(n_features):
            feature_values = dataset[:, feature_idx]
            thresholds = np.unique(feature_values)

            for threshold in thresholds:
                left_dataset, right_dataset = self.split(dataset, feature_idx, threshold)

                if len(left_dataset) > 0 and len(right_dataset) > 0:
                    y, left_y, right_y = dataset[:, -1], left_dataset[:, -1], right_dataset[:, -1]
                    
                    gain = self.calculate_gain(y, left_y, right_y)

                    if gain > max_gain:
                        best_split['feature_idx'] = feature_idx
                        best_split['threshold'] = threshold
                        best_split['gain'] = gain
                        best_split['left_dataset'] = left_dataset
                        best_split['right_dataset'] = right_dataset
                        max_gain = gain

        return best_split

    def split(self, dataset, feature_idx, threshold):
        left_dataset = dataset[dataset[:, feature_idx] <= threshold]
        right_dataset = dataset[dataset[:, feature_idx] > threshold]
        return left_dataset, right_dataset

    def calculate_gain(self, parent_y, left_y, right_y):
        weight_l = len(left_y) / len(parent_y)
        weight_r = len(right_y) / len(parent_y)

        if self.algorithm == 'CART':
            # Gini Gain
            p_gini = self.gini(parent_y)
            l_gini = self.gini(left_y)
            r_gini = self.gini(right_y)
            return p_gini - (weight_l * l_gini + weight_r * r_gini)

        elif self.algorithm == 'ID3':
            # Information Gain (Entropy)
            p_ent = self.entropy(parent_y)
            l_ent = self.entropy(left_y)
            r_ent = self.entropy(right_y)
            return p_ent - (weight_l * l_ent + weight_r * r_ent)

        elif self.algorithm == 'C4.5':
            # Gain Ratio
            p_ent = self.entropy(parent_y)
            l_ent = self.entropy(left_y)
            r_ent = self.entropy(right_y)
            info_gain = p_ent - (weight_l * l_ent + weight_r * r_ent)
            
            # Split Information
            split_info = -(weight_l * np.log2(weight_l + 1e-9) + weight_r * np.log2(weight_r + 1e-9))
            return info_gain / (split_info + 1e-9) # Avoid division by zero

    def entropy(self, y):
        class_labels = np.unique(y)
        ent = 0
        for cls in class_labels:
            p = len(y[y == cls]) / len(y)
            ent += -p * np.log2(p)
        return ent

    def gini(self, y):
        class_labels = np.unique(y)
        gini = 1
        for cls in class_labels:
            p = len(y[y == cls]) / len(y)
            gini -= p ** 2
        return gini

    def fit(self, X, y):
        start_time = time.time()
        dataset = np.column_stack((X, y))
        self.root = self.build_tree(dataset)
        train_time = time.time() - start_time
        return train_time

    def predict(self, X):
        return [self.predict_class(row, self.root) for row in X]

    def predict_class(self, row, node):
        if node.value is not None:
            return node.value
        if row[node.feature_idx] <= node.threshold:
            return self.predict_class(row, node.left)
        return self.predict_class(row, node.right)
        
    def get_depth(self, node=None):
        if node is None: node = self.root
        if node.value is not None: return 0
        return 1 + max(self.get_depth(node.left), self.get_depth(node.right))
        
    def get_n_leaves(self, node=None):
        if node is None: node = self.root
        if node.value is not None: return 1
        return self.get_n_leaves(node.left) + self.get_n_leaves(node.right)


# ==========================================
# 2. Evaluation Metrics (From Scratch)
# ==========================================
def calculate_metrics(y_true, y_pred):
    classes = np.unique(y_true)
    metrics = {'accuracy': 0, 'precision': 0, 'recall': 0, 'f1': 0}
    
    # Accuracy
    metrics['accuracy'] = np.mean(y_true == y_pred)
    
    # Macro-averaged Precision, Recall, F1
    precisions, recalls, f1s = [], [], []
    for c in classes:
        tp = np.sum((y_true == c) & (y_pred == c))
        fp = np.sum((y_true != c) & (y_pred == c))
        fn = np.sum((y_true == c) & (y_pred != c))
        
        precision = tp / (tp + fp) if (tp + fp) > 0 else 0
        recall = tp / (tp + fn) if (tp + fn) > 0 else 0
        f1 = 2 * (precision * recall) / (precision + recall) if (precision + recall) > 0 else 0
        
        precisions.append(precision)
        recalls.append(recall)
        f1s.append(f1)
        
    metrics['precision'] = np.mean(precisions)
    metrics['recall'] = np.mean(recalls)
    metrics['f1'] = np.mean(f1s)
    
    return metrics


# ==========================================
# 3. Execution and Comparison
# ==========================================
if __name__ == "__main__":
    # Load Iris dataset
    iris = load_iris()
    X, y = iris.data, iris.target
    X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

    algorithms = ['ID3', 'C4.5', 'CART']
    results = []

    for algo in algorithms:
        tree = DecisionTree(max_depth=5, algorithm=algo)
        
        # Train
        train_time = tree.fit(X_train, y_train)
        
        # Predict
        y_pred = tree.predict(X_test)
        
        # Evaluate metrics
        metrics = calculate_metrics(y_test, y_pred)
        
        # Tree Structure
        depth = tree.get_depth()
        leaves = tree.get_n_leaves()
        
        results.append({
            'Algorithm': algo,
            'Accuracy': f"{metrics['accuracy']:.4f}",
            'Precision': f"{metrics['precision']:.4f}",
            'Recall': f"{metrics['recall']:.4f}",
            'F1 Score': f"{metrics['f1']:.4f}",
            'Tree Depth': depth,
            'Leaf Nodes': leaves,
            'Train Time (s)': f"{train_time:.5f}"
        })

    # Print Results
    results_df = pd.DataFrame(results)
    print("\n--- Decision Tree Algorithm Comparison ---")
    print(results_df.to_string(index=False))



--- Decision Tree Algorithm Comparison ---
Algorithm Accuracy Precision Recall F1 Score  Tree Depth  Leaf Nodes Train Time (s)
      ID3   1.0000    1.0000 1.0000   1.0000           5           9        0.07633
     C4.5   1.0000    1.0000 1.0000   1.0000           5           9        0.06684
     CART   1.0000    1.0000 1.0000   1.0000           5           9        0.05243
